In [1]:
import pandas as pd
import numpy as np

In [2]:
Emicron = pd.read_csv("https://github.com/niconomist98/DataAnalyticsUQ/raw/refs/heads/main/Datos/EMICRON/emicron_identificacion.csv", 
                      encoding = "latin" , sep =",")

In [3]:
Emicron

,DIRECTORIO,SECUENCIA_P,SECUENCIA_ENCUESTA,COD_DEPTO,AREA,CLASE_TE,P35,P241,MES_REF,P3031,P3032_1,P3032_2,P3032_3,P3033,P3034,P3035,P3000,GRUPOS4,GRUPOS12,F_EXP
0,7627444,1,1,44,NaN,2,2,33,ENERO,2,NaN,NaN,NaN,2,240,2,2,2,3,60.050515
1,7627446,1,1,44,NaN,2,1,31,ENERO,2,NaN,NaN,NaN,2,166,2,2,1,1,86.341075
2,7627449,1,1,68,NaN,1,2,42,ENERO,2,NaN,NaN,NaN,2,60,2,2,3,5,139.884518
3,7627453,1,2,68,NaN,1,2,41,ENERO,2,NaN,NaN,NaN,2,60,2,2,4,7,168.635440
4,7627456,1,3,68,NaN,1,1,18,ENERO,2,NaN,NaN,NaN,2,9,2,2,4,6,64.659273
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78496,8038070,1,3,13,NaN,1,1,24,DICIEMBRE,2,NaN,NaN,NaN,2,48,1,2,4,7,362.509123
78497,8038071,1,1,13,NaN,2,1,30,DICIEMBRE,2,NaN,NaN,NaN,2,48,2,2,1,1,158.805116
78498,8038073,1,1,13,NaN,2,1,66,DICIEMBRE,2,NaN,NaN,NaN,2,500,2,2,1,1,57.451130
78499,8038074,1,3,13,NaN,2,1,71,DICIEMBRE,2,NaN,NaN,NaN,2,500,2,2,1,1,93.355386


In [4]:
#Pregunta 1 ¿Cuantos micronegocios hay en el Quindío, cuyos propietarios son hombres entre los 18 y 30 años?

#1. Revisar codificación de sexo
print("Valores de P35 (sexo):")
print(Emicron['P35'].value_counts(dropna=False))

#2. Filtrar Quindío, hombres, 18–30 años 

df_q = Emicron[
    (Emicron['COD_DEPTO'] == 63) &    # Quindío
    (Emicron['P35'] == 1) &           # Hombres (verifica con value_counts)
    (Emicron['P241'].between(18,30))  # Edad entre 18 y 30
].copy()

#3. Resultados
no_ponderado = df_q.shape[0]
ponderado = df_q['F_EXP'].sum()

#Clara
print("Pregunta 1:")
print("¿Cuántos micronegocios hay en el Quindío, cuyos propietarios son hombres entre los 18 y 30 años?")
print(f"➡️ Respuesta (usando factor de expansión): {ponderado:,.0f} micronegocios")
print(f"(Registros observados en la muestra: {no_ponderado})")

Valores de P35 (sexo):
P35
1    49085
2    29416
Name: count, dtype: int64
Pregunta 1:
¿Cuántos micronegocios hay en el Quindío, cuyos propietarios son hombres entre los 18 y 30 años?
➡️ Respuesta (usando factor de expansión): 1,929 micronegocios
(Registros observados en la muestra: 129)


In [5]:
#Pregunta 2. Cuantos micronegocios de la industria manufacturera existen el departamento de Risaralda cuyas propietarias son mujeres entre los 20 y 30 años.

#1. Filtrar Risaralda, mujeres, 20–30 años, sector manufacturero
df_ris = Emicron[
    (Emicron['COD_DEPTO'] == 66) &     # Risaralda
    (Emicron['P35'] == 2) &            # Mujeres
    (Emicron['P241'].between(20,30)) & # Edad 20–30
    (Emicron['GRUPOS12'] == 3)         # Industria manufacturera
].copy()

#2. Resultados
no_ponderado = df_ris.shape[0]
ponderado = df_ris['F_EXP'].sum()

#3. Respuesta Clara
print("Pregunta 2:")
print("¿Cuántos micronegocios de la industria manufacturera existen en Risaralda cuyas propietarias son mujeres entre 20 y 30 años?")
print(f"➡️ Respuesta (usando factor de expansión): {ponderado:,.0f} micronegocios")
print(f"(Registros observados en la muestra: {no_ponderado})")

Pregunta 2:
¿Cuántos micronegocios de la industria manufacturera existen en Risaralda cuyas propietarias son mujeres entre 20 y 30 años?
➡️ Respuesta (usando factor de expansión): 471 micronegocios
(Registros observados en la muestra: 16)


In [6]:
#Pregunta 3. Entre las ciudades de Manizales, Pereira y Armenia ¿cual de las tres ciudades tiene más micronegocios, del sector comercio cuyas propietarias son mujeres, mayores a 40 años?

# Códigos DANE para AREA:
# Manizales = 17, Pereira = 66, Armenia = 63
ciudades = [17, 66, 63]

df_cities = Emicron[
    (Emicron['AREA'].isin(ciudades)) &  # usamos AREA
    (Emicron['P35'] == 2) &             # Mujeres
    (Emicron['P241'] > 40) &            # Mayores de 40 años
    (Emicron['GRUPOS12'] == 5)          # Comercio
].copy()

# Conteo por ciudad
resumen = df_cities.groupby('AREA').agg(
    Registros=('DIRECTORIO','count'),
    Micronegocios=('F_EXP','sum')
).reset_index()

# Mapear nombres de ciudades
ciudades_map = {17:'Manizales', 66:'Pereira', 63:'Armenia'}
resumen['Ciudad'] = resumen['AREA'].map(ciudades_map)

# Ordenar por micronegocios ponderados
resumen = resumen.sort_values('Micronegocios', ascending=False)

print("Pregunta 3:")
print("Entre Manizales, Pereira y Armenia, ¿cuál tiene más micronegocios del sector comercio cuyas propietarias son mujeres mayores de 40 años?")
print(resumen)

# Mostrar ciudad con mayor número
mayor = resumen.iloc[0]
print(f"\n➡️ Respuesta: {mayor['Ciudad']} con {mayor['Micronegocios']:,.0f} micronegocios")

Pregunta 3:
Entre Manizales, Pereira y Armenia, ¿cuál tiene más micronegocios del sector comercio cuyas propietarias son mujeres mayores de 40 años?
   AREA  Registros  Micronegocios     Ciudad
2  66.0        166    3827.329750    Pereira
0  17.0        163    2636.852997  Manizales
1  63.0        192    2557.641662    Armenia

➡️ Respuesta: Pereira con 3,827 micronegocios


In [7]:
#Pregunta 4. En la ciudad de Armenia cuantos micronegocios cuyos propietarios son hombres que no tienen nombre comercial el micronegocio.
# Filtrar Armenia, hombres, sin nombre comercial ---
df_armenia = Emicron[
    (Emicron['AREA'] == 63) &      # Armenia
    (Emicron['P35'] == 1) &        # Hombres
    (Emicron['P3035'] == 2)        # Sin nombre comercial
].copy()

#Resultados
no_ponderado = df_armenia.shape[0]
ponderado = df_armenia['F_EXP'].sum()

#Respuesta Clara
print("Pregunta 4:")
print("En la ciudad de Armenia, ¿cuántos micronegocios cuyos propietarios son hombres no tienen nombre comercial?")
print(f"➡️ Respuesta (usando factor de expansión): {ponderado:,.0f} micronegocios")
print(f"(Registros observados en la muestra: {no_ponderado})")

Pregunta 4:
En la ciudad de Armenia, ¿cuántos micronegocios cuyos propietarios son hombres no tienen nombre comercial?
➡️ Respuesta (usando factor de expansión): 11,603 micronegocios
(Registros observados en la muestra: 822)


In [12]:
#cual es el departamento de colombia que tiene mayor cantidad de micronegocios cuyas dueñas son mujeres

micronegocios_por_dpto = Emicron.groupby(["COD_DEPTO","P35"])["F_EXP"].sum()
micronegocios_por_dpto

COD_DEPTO  P35
5          1      401961.465988
           2      228065.054284
8          1      203052.166264
           2      127459.192128
11         1      370414.355785
           2      218597.578012
13         1      237817.472299
           2       89256.882936
15         1       64788.375168
           2       47304.823258
17         1       57810.647554
           2       27679.193558
18         1       33905.164393
           2       15187.450504
19         1      153569.363753
           2       87585.021376
20         1      100135.189800
           2       53543.138605
23         1      212544.942870
           2       99508.457678
25         1      152633.806724
           2       68092.478848
27         1       21019.762856
           2       10876.912625
41         1       56820.698162
           2       30260.144557
44         1       69470.742444
           2       51573.204613
47         1      158680.430585
           2       61369.107902
50         1       63151.

In [14]:
grupos12_map = {
    1: "Agricultura, ganadería, caza, silvicultura y pesca",
    2: "Minería",
    3: "Industria manufacturera",
    4: "Construcción",
    5: "Comercio y reparación de vehículos automotores y motocicletas",
    6: "Transporte y almacenamiento",
    7: "Alojamiento y servicios de comida",
    8: "Información y comunicaciones",
    9: "Actividades inmobiliarias, profesionales y servicios administrativos",
    10: "Educación",
    11: "Actividades de atención a la salud humana y de asistencia social",
    12: "Actividades artísticas, entretenimiento, recreación y otros servicios"
}

In [22]:
#cuantos micronegocios hay en cada uno de las ramas de actividad en el quindío (GRUPOS12)

quindio = Emicron[Emicron["COD_DEPTO"] == 63]

ramas_quindio = quindio.groupby("GRUPOS12")["F_EXP"].sum().reset_index()

ddf = pd.DataFrame(list(grupos12_map.items()), columns=["GRUPOS12", "Rama_Actividad"])

ramas_quindio = ramas_quindio.merge(ddf, on="GRUPOS12", how="left")

ramas_quindio = ramas_quindio.sort_values("F_EXP", ascending=False)

print("Micronegocios por rama de actividad en el Quindío:")
print(ramas_quindio)

Micronegocios por rama de actividad en el Quindío:
    GRUPOS12         F_EXP                                     Rama_Actividad
4          5  12297.769145  Comercio y reparación de vehículos automotores...
11        12   5903.673223  Actividades artísticas, entretenimiento, recre...
6          7   5063.944221                  Alojamiento y servicios de comida
5          6   3321.986172                        Transporte y almacenamiento
2          3   3196.162878                            Industria manufacturera
8          9   2844.505269  Actividades inmobiliarias, profesionales y ser...
3          4   2302.749621                                       Construcción
0          1    990.702778  Agricultura, ganadería, caza, silvicultura y p...
10        11    471.451206  Actividades de atención a la salud humana y de...
7          8    444.926626                       Información y comunicaciones
9         10    309.064230                                          Educación
1          2 